In [58]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

True

Ingestion → Chunking → Embeddings → Vector Store → Retrieval → Context Injection → LLM

In [59]:
from langchain_community.document_loaders import TextLoader
import os

def load_documents():
    loader = TextLoader("data/notes.txt")   # replace with your files
    return loader.load()

In [60]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_documents(docs):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=100
    )

    return splitter.split_documents(docs)

In [61]:
from langchain_openai import OpenAIEmbeddings

def create_embeddings():
    return OpenAIEmbeddings(model="text-embedding-3-small",
                                openai_api_key=os.getenv("OPENAI_API_KEY"))

In [62]:
from langchain_community.vectorstores import Chroma

def create_vector_store(chunks, embeddings):

    return Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name="rag_demo"
    )

In [63]:
from langchain_community.vectorstores import Chroma

def create_vector_store(chunks, embeddings):

    return Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name="rag_demo"
    )

In [64]:
def build_context(docs):

    return "\n\n".join([doc.page_content for doc in docs])

In [65]:
def retrieve_context(vector_store, query, k=3):

    return vector_store.similarity_search(query, k=k)

In [66]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

def generate_answer(context, query):

    llm = ChatOpenAI(model="gpt-4o", temperature=0)

    messages = [
        SystemMessage(content="You are a helpful assistant."),
        HumanMessage(content=f"""
Use the following context to answer the question.

Context:
{context}

Question:
{query}
""")
    ]

    return llm.invoke(messages).content

In [67]:
def rag_pipeline(query):

    # Ingestion
    docs = load_documents()

    # Chunking
    chunks = chunk_documents(docs)

    # Embeddings
    embeddings = create_embeddings()

    # Storage
    vector_store = create_vector_store(chunks, embeddings)

    # Retrieval
    retrieved_docs = retrieve_context(vector_store, query)

    # Context Injection
    context = build_context(retrieved_docs)

    # LLM
    answer = generate_answer(context, query)

    return answer

In [68]:
query = "What pricing strategy works best for enterprise customers?"

response = rag_pipeline(query)

print(response)

The best pricing strategy for enterprise customers is tiered pricing that emphasizes reliability, security, and support. Enterprise customers are willing to pay higher prices for features such as guaranteed uptime and dedicated account management. Therefore, a tiered pricing model that offers these premium features and support levels would be most effective for targeting enterprise customers.
